# CO₂ Storage Capacity Assessment

This notebook runs a probabilistic static storage-capacity assessment using:

$$SC = GRV \times (N/G) \times \phi \times \rho_{CO_2} \times S_{eff}$$

Change the values in the **Editable inputs** cell, then choose **Runtime → Run all**. The example values represent the Rødby Bunter Sandstone assessment.

In [ ]:
# Install the latest package and plotting tools from GitHub.
%pip install -q "git+https://github.com/AnaSoles/ggg-co2-storage-eval.git" matplotlib pandas

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from storageeval import Distribution, StorageSite, simulate

plt.style.use("seaborn-v0_8-whitegrid")

## Editable inputs

Enter minimum, most likely, and maximum values. Fractions such as porosity must be decimals: `0.23` means 23%.

In [ ]:
site_name = "Rødby – Bunter Sandstone"
iterations = 100_000
random_seed = 42

#                       minimum, most likely, maximum
grv_km3 =               (22.57, 28.21, 33.85)
net_to_gross =          (0.20,  0.25,  0.30)
porosity =              (0.184, 0.23,  0.276)
co2_density_kg_m3 =     (573.4, 603.6, 764.0)
storage_efficiency =    (0.05,  0.10,  0.20)

In [ ]:
site = StorageSite(
    name=site_name,
    grv=Distribution.pert(*grv_km3),
    net_to_gross=Distribution.pert(*net_to_gross),
    porosity=Distribution.pert(*porosity),
    co2_density=Distribution.pert(*co2_density_kg_m3),
    storage_efficiency=Distribution.pert(*storage_efficiency),
)

result = simulate(site, iterations=iterations, seed=random_seed)
summary = result.summary()
pd.DataFrame(
    {"Capacity (Mt CO₂)": [summary["p90_mt"], summary["p50_mt"], summary["p10_mt"], summary["mean_mt"]]},
    index=["P90 (conservative)", "P50 (median)", "P10 (upside)", "Mean"],
).round(2)

## Input uncertainty distributions

In [ ]:
labels = {
    "grv_km3": "GRV (km³)",
    "net_to_gross": "Net-to-gross",
    "porosity": "Porosity",
    "co2_density_kg_m3": "CO₂ density (kg/m³)",
    "storage_efficiency": "Storage efficiency",
}
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, (name, values) in zip(axes.flat, result.inputs.items()):
    ax.hist(values, bins=45, color="#2a6fbb", alpha=0.82)
    ax.set_title(labels[name])
    ax.set_ylabel("Simulations")
axes.flat[-1].axis("off")
fig.suptitle(f"{site_name} – input uncertainty", fontsize=15)
fig.tight_layout()
plt.show()

## Storage-capacity probability distribution

In [ ]:
fig, ax = result.plot_distribution()
for key, color in [("p90_mt", "#c75146"), ("p50_mt", "#333333"), ("p10_mt", "#2a6fbb")]:
    ax.axvline(summary[key], color=color, linestyle="--", label=f"{key[:3].upper()}: {summary[key]:.1f} Mt")
ax.legend()
fig.set_size_inches(10, 6)
plt.show()

## Exceedance curve

P90 is the capacity that has a 90% probability of being exceeded; P10 is the upside estimate.

In [ ]:
fig, ax = result.plot_exceedance()
for probability, key in [(90, "p90_mt"), (50, "p50_mt"), (10, "p10_mt")]:
    ax.scatter(summary[key], probability, s=55, label=f"P{probability}: {summary[key]:.1f} Mt")
ax.legend()
fig.set_size_inches(10, 6)
plt.show()

## Sensitivity tornado chart

Longer bars identify the assumptions with the strongest influence on calculated capacity.

In [ ]:
fig, ax = result.plot_sensitivity()
fig.set_size_inches(10, 6)
plt.show()

## Important limitation

This is a static volumetric screening assessment. It does not yet represent pressure constraints, injectivity, plume migration, dynamic reservoir simulation, or economics.